# Flight Ticket Price Prediction - Data Preprocessing & Feature Engineering

## Objective

The objective of this notebook is to transform the raw dataset into a machine learning ready dataset.

In this notebook, we will:

- Assess data quality
- Handle missing values (if any)
- Remove duplicate records
- Perform feature engineering
- Select useful features
- Encode categorical variables
- Scale numerical features
- Build a preprocessing pipeline

The final output of this notebook will be a clean and processed dataset that can be directly used for machine learning model training.

In [2]:
# =====================================================
# Import Required Libraries
# =====================================================

# Data Manipulation
import pandas as pd
import numpy as np

# Data Visulization (for quick verification)
import matplotlib.pyplot as plt
import seaborn as sns

# machine learning preprocessing 
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import(
    OneHotEncoder,
    StandardScaler
)

# Save Objects
import joblib

# ignore Warnings
import warnings
warnings.filterwarnings("ignore")

# Display Settings
pd.set_option("display.max_columns", None)

print("Librarries Imported Successfully.")

Librarries Imported Successfully.


# Load Dataset

## Objective

In this step, we load the cleaned flight ticket dataset and create a separate working copy for preprocessing.

Creating a working copy ensures that the original dataset remains unchanged throughout the preprocessing pipeline.

This is considered a best practice in machine learning projects because it preserves the original data and makes experiments reproducible.

In [3]:
# =====================================================
# Load Dataset
# =====================================================

# Read Dataset
df = pd.read_csv("Dataset/Clean_Dataset.csv")

# Create Working Copy
data = df.copy()

# Display Dataset Information
print(f"Dataset Shape : {data.shape}")

display(data.head())

Dataset Shape : (300153, 12)


,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955


# Data Quality Assessment

## Objective

Before applying any preprocessing techniques, it is important to evaluate the quality of the dataset.

This assessment helps us to identify:

- Dataset dimensions
- Data types
- Missing values
- Duplicate records
- Unique values
- Memory usage

A proper data quality assessment ensures that preprocessing decisions are based on evidence rather than assumptions.

In [4]:
# =====================================================
# Data Quality Assessment
# =====================================================

# Dataset Summary
summary = pd.DataFrame({

    "Data Type": data.dtypes,

    "Missing Values": data.isnull().sum(),

    "Missing (%)": (
        data.isnull().sum() / len(data) * 100).round(2),

    "Unique Values": data.nunique(),

    "Duplicate Rows": data.duplicated().sum(),

    "Memory Usage (KB)": (
        data.memory_usage(deep=True) / 1024).round(2)

})

display(summary)

print("=" * 70)
print(f"Dataset Shape       :{data.shape}")
print(f"Total Missing       :{data.isnull().sum().sum()}")
print(f"Duplicate Rows      :{data.duplicated().sum()}")
print(f"Memory Usage (MB)   :{data.memory_usage(deep=True).sum()/1024**2:.2f}")


,Data Type,Missing Values,Missing (%),Unique Values,Duplicate Rows,Memory Usage (KB)
Index,NaN,NaN,NaN,NaN,0,0.13
Unnamed: 0,int64,0.0,0.0,300153.0,0,2344.95
airline,object,0.0,0.0,6.0,0,16561.93
arrival_time,object,0.0,0.0,6.0,0,16441.67
class,object,0.0,0.0,2.0,0,16505.91
days_left,int64,0.0,0.0,49.0,0,2344.95
departure_time,object,0.0,0.0,6.0,0,16809.36
destination_city,object,0.0,0.0,6.0,0,16428.07
duration,float64,0.0,0.0,476.0,0,2344.95
flight,object,0.0,0.0,1561.0,0,16164.22


Dataset Shape       :(300153, 12)
Total Missing       :0
Duplicate Rows      :0
Memory Usage (MB)   :136.81


# Data Cleaning Decisions

## Objective

Before applying any preprocessing techniques, we validate the results of the data quality assessment and decide whether any cleaning operation is required.

The following checks will be performed:

- Missing Values
- Duplicate Records
- Data Types
- Unnecessary Columns

Instead of blindly applying preprocessing, we make decisions based on the actual condition of the dataset.

In [5]:
# =====================================================
# Data Cleaning Decisions
# =====================================================

# Missing Values
missing = data.isnull().sum().sum()

if missing == 0:
    print("✅ No Missing Values Found.")
else:
    print(f"⚠ Total Missing Values : {missing}")

print("-"*60)

# Duplicate Rows
duplicates = data.duplicated().sum()

if duplicates == 0:
    print("✅ No Duplicae Rows Found.")
else:
    print(f"⚠ Duplicate Rows : {duplicates}")

print("-"*60)

# Data Types
print("Data Types")
display(data.dtypes.to_frame(name="Data Type"))

print("-"*60)

# Unnecessary Columns
if "Unnamed: 0" in data.columns:

    print("Removing 'Unnamed: 0' Column...")

    data.drop(columns="Unnamed: 0", inplace=True)

    print("✅ Column Removed Sucessfully.")

else:

    print("✅ No Unnecessary Columns Found.")

✅ No Missing Values Found.
------------------------------------------------------------
✅ No Duplicae Rows Found.
------------------------------------------------------------
Data Types


,Data Type
Unnamed: 0,int64
airline,object
flight,object
source_city,object
departure_time,object
stops,object
arrival_time,object
destination_city,object
class,object
duration,float64


------------------------------------------------------------
Removing 'Unnamed: 0' Column...
✅ Column Removed Sucessfully.


# Feature Selection

## Objective

Feature selection is the process of selecting only those features that are useful for predicting the target variable.

In this step, we will:

- Identify useful features.
- Remove unnecessary features.
- Separate input features (X) and target variable (y).

Selecting the right features improves:

- Model performance
- Training speed
- Model interpretability
- Generalization on unseen data

In [6]:
# =====================================================
# Feature Selection
# =====================================================

# Drop unnecessary column
data = data.drop(columns=["flight"])

# Features (X)
X = data.drop(columns=["price"])

# Target (y)
y = data["price"]

print(f"Feature Matrix shape : {X.shape}")
print(f"Target Shape         : {y.shape}")

print("\nSelected Features:")
print(X.columns.tolist())

Feature Matrix shape : (300153, 9)
Target Shape         : (300153,)

Selected Features:
['airline', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'class', 'duration', 'days_left']


# Train-Test Split

## Objective

Before applying preprocessing techniques, we split the dataset into training and testing sets.

This is an essential step because preprocessing should be learned **only from the training data** and then applied to the test data.

This approach helps to:

- Prevent data leakage
- Evaluate model performance fairly
- Simulate real-world prediction scenarios

In this project, we use an **80:20 split**, where:

- **80%** of the data is used for training.
- **20%** of the data is reserved for testing.

In [7]:
# =====================================================
# Train-Test Split
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("=" * 60)

print(f"Training features : {X_train.shape}")
print(f"Testing features  : {X_test.shape}")

print(f"Training Target : {y_train.shape}")
print(f"Testing Target  : {y_test.shape}")

Training features : (240122, 9)
Testing features  : (60031, 9)
Training Target : (240122,)
Testing Target  : (60031,)


# Build the Preprocessing Pipeline

## Objective

Machine learning models require numerical input. Therefore, categorical and numerical features must be preprocessed before model training.

In this step, we create a preprocessing pipeline using **ColumnTransformer**.

The preprocessing strategy is:

### Categorical Features
- One-Hot Encoding

### Numerical Features
- Standard Scaling

Using a preprocessing pipeline ensures:

- No data leakage
- Reusable preprocessing
- Production-ready workflow
- Easy integration with machine learning models

In [8]:
# =====================================================
# Build Preprocessing Pipeline
# =====================================================

# Categorical Features
Categorical_featues = [
    "airline",
    "source_city",
    "departure_time",
    "stops",
    "arrival_time",
    "destination_city",
    "class"
]

# Numerical Features
numerical_features = [
    "duration",
    "days_left"
]

# Column Transformer
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            Categorical_featues
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

print("✅ Preprocessing Pipeline Created Sucessfully.")

✅ Preprocessing Pipeline Created Sucessfully.


# Apply the Preprocessing Pipeline

## Objective

In this step, we apply the preprocessing pipeline to both the training and testing datasets.

The preprocessing workflow is:

- Fit the preprocessing pipeline on the training dataset.
- Transform the training dataset.
- Transform the testing dataset using the same fitted pipeline.

This ensures that the model learns preprocessing parameters only from the training data, preventing data leakage.

Finally, we inspect the transformed feature matrix and generated feature names.

In [9]:
# =====================================================
# Apply Preprocessing Pipeline
# =====================================================

# Fit on training data and transform
X_train_processed = preprocessor.fit_transform(X_train)

# Transform testing data
X_test_processed = preprocessor.transform(X_test)

# Feature Names
feature_names = preprocessor.get_feature_names_out()

print("=" * 60)

print(f"training shape  : {X_train_processed.shape}")
print(f"Testing Shape  : {X_test_processed.shape}")

print(f"\nTotal Features After Encoding :  {len(feature_names)}")

print("\nFirst 15 Feature Names:")

for feature in feature_names[:15]:
    print(feature)


training shape  : (240122, 37)
Testing Shape  : (60031, 37)

Total Features After Encoding :  37

First 15 Feature Names:
categorical__airline_AirAsia
categorical__airline_Air_India
categorical__airline_GO_FIRST
categorical__airline_Indigo
categorical__airline_SpiceJet
categorical__airline_Vistara
categorical__source_city_Bangalore
categorical__source_city_Chennai
categorical__source_city_Delhi
categorical__source_city_Hyderabad
categorical__source_city_Kolkata
categorical__source_city_Mumbai
categorical__departure_time_Afternoon
categorical__departure_time_Early_Morning
categorical__departure_time_Evening


# Save the Preprocessing Pipeline

## Objective

After building the preprocessing pipeline, we save it as a serialized object using **Joblib**.

Saving the preprocessor ensures that the exact same preprocessing steps can be applied to new data during:

- Model Training
- Model Evaluation
- Deployment
- Production Inference

This guarantees consistency between training data and real-world prediction data.

In [10]:
# =====================================================
# Save Preprocessing Pipeline
# =====================================================

import os
import joblib

# Create artifacts folder if it doesn't exist
os.makedirs("../artifacts", exist_ok=True)

# Save the preprocessor
joblib.dump(preprocessor, "artifacts/preprocessor.pkl")

print("✅ Preprocessing Pipeline Saved Successfully.")
print("Location : artifacts/preprocessor.pkl")

✅ Preprocessing Pipeline Saved Successfully.
Location : artifacts/preprocessor.pkl


# Notebook Summary

## Overview

This notebook focused on preparing the flight ticket dataset for machine learning by applying industry-standard preprocessing techniques.

---

## Tasks Completed

### 1. Data Loading
- Loaded the cleaned dataset.
- Created a working copy for preprocessing.

### 2. Data Quality Assessment
- Verified dataset dimensions.
- Checked data types.
- Checked missing values.
- Checked duplicate records.
- Verified memory usage.

### 3. Data Cleaning
- Confirmed no missing values.
- Confirmed no duplicate rows.
- Removed unnecessary columns (if present).

### 4. Feature Selection
- Removed non-informative features.
- Separated input features (X) and target variable (y).

### 5. Train-Test Split
- Split the dataset into:
  - 80% Training Data
  - 20% Testing Data

### 6. Preprocessing Pipeline
Created an industry-standard preprocessing pipeline using:

- ColumnTransformer
- OneHotEncoder
- StandardScaler

### 7. Feature Transformation
- Applied preprocessing on the training dataset.
- Applied the same transformation on the testing dataset.
- Prevented data leakage by fitting only on training data.

### 8. Pipeline Serialization
- Saved the preprocessing pipeline using Joblib.
- Generated:
  - artifacts/preprocessor.pkl

---

## Output of this Notebook

After completing this notebook, we have:

- Clean Training Features
- Clean Testing Features
- Training Target
- Testing Target
- Saved Preprocessing Pipeline

These outputs are now ready for machine learning model training.

---

## Key Learning Outcomes

- Data Quality Assessment
- Feature Selection
- Train-Test Split
- One-Hot Encoding
- Feature Scaling
- ColumnTransformer
- Pipeline Creation
- Data Leakage Prevention
- Joblib Serialization

---

## Next Notebook

Notebook 4:
**Model Building & Model Selection**

In the next notebook, we will:

- Train multiple regression models
- Compare model performance
- Perform Hyperparameter Tuning
- Select the Best Model
- Save the Best Model